# Step 2: Competitive Landscape

## 2.1 Basic Imports and Data Loading

In [97]:
import re
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import PercentFormatter
from scipy import stats                      # linear regression (Lotka) and Kendall's tau
from sqlalchemy import func, or_

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,               # applications
    TLS202_APPLN_TITLE,         # titles
    TLS203_APPLN_ABSTR,         # abstracts
    TLS206_PERSON,              # persons: standardised name, sector, country
    TLS207_PERS_APPLN,          # person <-> application links (applicant / inventor)
    TLS209_APPLN_IPC,           # IPC classification
    TLS211_PAT_PUBLN,           # publications: grants and their dates
    TLS224_APPLN_CPC,           # CPC classification
)

In [98]:
# ---------------------------------------------------------------- environment
ENV = "TEST"                         # develop on TEST, switch to "PROD" for the final numbers

# ---------------------------------------------------------------- applicant names (Section 2)
NAME_METHOD = "dictionary"           # "psn" (PATSTAT names as they are), "dictionary" or "cleanco"
GOLD_STANDARD_FILE = "PATSTAT_500_label_datav4.csv"   # hand-labelled names (Section 2.4), in the same folder as this notebook

# ---------------------------------------------------------------- replication (Section 4): the paper's design
REPL_OFFICE = "US"                                      # USPTO patents...
REPL_GRANTED = (date(1976, 1, 1), date(2002, 12, 31))   # ...granted between these two dates
REPL_KEYWORDS = ("fuel cell",)                          # searched in titles and abstracts
REPL_CLASSES = ("H01M8",)                               # IPC/CPC H01M 8: fuel cells
REPL_SELECTION = "keywords or classes"                  # "keywords", "classes" or "keywords or classes"
PAPER_COUNTRIES = ("AU", "CA", "FR", "DE", "IL", "IT", "JP", "NL", "SE", "CH", "GB", "US")

# ---------------------------------------------------------------- your sector (Section 5): replace with your Step 1
SECTOR_NAME = "Surface Drones"

SECTOR_CPC_PREFIXES = (
    # Primary classification codes for Unmanned Surface Vessels (USVs)
    "B63B2035/006",  # Unmanned surface vessels, e.g. remotely controlled
    "B63B2035/007",  # Unmanned surface vessels, autonomously operating
    "B63B2035/008",  # Unmanned surface vessels, remotely controlled

    # Vessel Control, Navigation, and Autonomy
    "G05D1/00",      # Control of position, course, altitude or attitude of land, water, air, or space vehicles (Autopilot/Autonomous navigation)
    "G05D1/0206",    # Control of position or course of waterborne vessels
    "G08G3/00",      # Traffic control systems for marine craft

    # Waterborne craft adapted for special or military purposes
    "B63B35/00",     # Vessels or floating structures specially adapted for specific purposes
    "B63G8/001",     # Unmanned underwater or surface-supported craft systems / deployment
    "B63G13/00",     # Defense or military features adapted for waterborne vessels

    # Hybrid & amphibious drone operations (Air-Sea interactions)
    "B64U60/10",     # UAVs specially adapted for use on water (water-landing/amphibious drones)
    "B64U70/93"      # Launching/landing UAV platforms for use on nautical vehicles
)

SECTOR_CPC_PREFIXES = (
    # Primary classification codes for Unmanned Surface Vessels (USVs)
    "B63B2035/006",  # Unmanned surface vessels, e.g. remotely controlled
    "B63B2035/007",  # Unmanned surface vessels, autonomously operating
    "B63B2035/008",  # Unmanned surface vessels, remotely controlled

    # Vessel Control, Navigation, and Autonomy
    "G05D1/00",      # Control of position, course, altitude or attitude of land, water, air, or space vehicles (Autopilot/Autonomous navigation)
    "G05D1/0206",    # Control of position or course of waterborne vessels
    "G08G3/00",      # Traffic control systems for marine craft

    # Waterborne craft adapted for special or military purposes
    "B63B35/00",     # Vessels or floating structures specially adapted for specific purposes
    "B63G8/001",     # Unmanned underwater or surface-supported craft systems / deployment
    "B63G13/00",     # Defense or military features adapted for waterborne vessels

    # Hybrid & amphibious drone operations (Air-Sea interactions)
    "B64U60/10",     # UAVs specially adapted for use on water (water-landing/amphibious drones)
    "B64U70/93"      # Launching/landing UAV platforms for use on nautical vehicles
)

# 1. Primary Vehicle Terminology
keywords_primary = [
    "unmanned surface vehicle",
    "USV",
    "unmanned surface vessel",
    "autonomous surface vehicle",
    "ASV",
    "autonomous surface vessel",
    "unmanned surface drone",
    "maritime drone",
    "marine drone",
    "aquatic drone",
    "waterborne drone",
    "remotely operated surface vessel",
    "crewless vessel",
    "uncrewed surface vessel",
    "robotic boat",
    "robotic vessel",
    "autonomous boat",
]

# 2. Form Factor & Design Terms
keywords_form_factor = [
    "surface craft",
    "surface vessel",
    "surface platform",
    "unmanned catamaran",
    "unmanned trimaran",
    "unmanned kayak",
    "semi-submersible surface vehicle",
    "autonomous hydrofoil",
    "wave glider",
    "saildrone",
]

# 3. Subsystem & Functional Keywords
keywords_subsystems = {
    "navigation_autonomy": [
        "obstacle avoidance",
        "path planning",
        "colregs compliance",
        "collision avoidance",
        "waypoint navigation",
        "autonomous docking",
        "dynamic positioning",
    ],
    "control_telemetry": [
        "teleoperation",
        "line-of-sight communication",
        "satellite communication",
        "SATCOM",
        "tethered control",
        "ground control station",
        "GCS",
    ],
    "propulsion_energy": [
        "hybrid propulsion",
        "electric outboard",
        "wave propulsion",
        "solar sails",
        "rigid sail",
        "waterjet propulsion",
        "thruster configuration",
    ],
    "payload_deployment": [
        "LARS",
        "launch and recovery system",
        "moon pool",
        "tether winch",
        "bathymetric sonar",
        "multibeam echosounder",
        "ROV deployment",
        "UAV landing pad",
    ],
}

SECTOR_KEYWORDS = (
    keywords_primary
    + keywords_form_factor
    + [kw for sublist in keywords_subsystems.values() for kw in sublist]
)

YEAR_MIN, YEAR_MAX = 2000, 2025
LAST_COMPLETE_YEAR = YEAR_MAX - 3    # the last 3 priority years are still incomplete in PATSTAT
IPR_TYPE = "PI"                      # patents of invention only
SPLIT_YEAR = 2015                    # incumbents file before SPLIT_YEAR, entrants from SPLIT_YEAR on
WINDOW = 3                           # years per window when following concentration over time
TOP_N = 15                           # rows in the tables of top applicants

COUNTRY_NAMES = {"AU": "Australia", "CA": "Canada", "FR": "France", "DE": "Germany", "IL": "Israel", "IT": "Italy",
                 "JP": "Japan", "NL": "Netherlands", "SE": "Sweden", "CH": "Switzerland", "GB": "UK", "US": "USA"}

In [99]:
patstat = PatstatClient(env=ENV)
db = patstat.orm()

## 2. The unit of analysis: applicants
In Step 1 we counted families and applications. 
Concentration needs one more unit: the applicant, the owner of the patent.

We must decide when two names on two patents are the same applicant.

The same company appears in PATSTAT under many spellings (SIEMENS AG, Siemens Aktiengesellschaft, SIEMENS AKTIENGESELLSCHAFT MUENCHEN…). If we count each spelling as a different firm, a large firm is split into many small ones and every concentration measure is wrong. tls206_person offers:

psn_name is a good start, but it still leaves variants (TOYOTA JIDOSHA KK and TOYOTA MOTOR CORP, with and without legal forms…), and it never merges a subsidiary with its parent.

To address the issue, we'll rely on **cleanco** package:

In [100]:
%pip install -q cleanco
from cleanco import basename


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Cleaning Function

In [101]:
# Legal-form words to remove (extend the set for the countries of your sector)
LEGAL_FORMS = {
    "INC", "INCORPORATED", "CORP", "CORPORATION", "CO", "COMPANY", "LTD", "LIMITED", "LLC", "LP", "PLC",
    "GMBH", "AG", "KG", "MBH", "SE", "SA", "SAS", "SPA", "SRL", "BV", "NV", "AB", "AS", "ASA", "OY", "OYJ",
    "KK", "KABUSHIKI", "KAISHA", "LTDA", "PTY", "THE", "AKTIENGESELLSCHAFT", "GESELLSCHAFT",
}

def clean_with_dictionary(name):
    "Upper-case, keep letters and digits only, and drop legal-form words at the start and at the end."
    words = re.sub(r"[^A-Z0-9 ]", " ", str(name).upper()).split()
    while words and words[-1] in LEGAL_FORMS:      # e.g. "TOYOTA JIDOSHA KABUSHIKI KAISHA" -> "TOYOTA JIDOSHA"
        words.pop()
    while words and words[0] in LEGAL_FORMS:       # e.g. "KABUSHIKI KAISHA TOSHIBA" -> "TOSHIBA"
        words.pop(0)
    return " ".join(words)

def clean_with_cleanco(name):
    "Remove legal forms with cleanco, then apply the same upper-case cleaning as method A."
    return clean_with_dictionary(basename(str(name)))

In [102]:
def keyword_condition(terms):
    """OR-condition: any term appears in the title or in the abstract (case-insensitive)."""
    title = func.lower(TLS202_APPLN_TITLE.appln_title)
    abstract = func.lower(TLS203_APPLN_ABSTR.appln_abstract)
    return or_(
        *(title.like(f"%{t.lower()}%") for t in terms),
        *(abstract.like(f"%{t.lower()}%") for t in terms),
    )


def symbol_prefix_condition(symbol_column, codes):
    """OR-condition: the blank-stripped classification symbol starts with one of the codes."""
    symbol = func.replace(symbol_column, " ", "")
    return or_(*(symbol.like(code.replace(" ", "") + "%") for code in codes))


# Base columns shown for every application
APPLN_COLUMNS = (
    TLS201_APPLN.appln_id,
    TLS201_APPLN.appln_auth,
    TLS201_APPLN.appln_nr,
    TLS201_APPLN.appln_kind,
    TLS201_APPLN.appln_filing_date,
)


def query_by_keywords(*columns):
    """Applications whose title or abstract contains a surface drone keyword."""
    
    """Note: an application might have a title but not an abstract or vice-versa"""
    
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .outerjoin(TLS202_APPLN_TITLE, TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id)
        .outerjoin(TLS203_APPLN_ABSTR, TLS201_APPLN.appln_id == TLS203_APPLN_ABSTR.appln_id)
        .filter(keyword_condition(SECTOR_KEYWORDS))
    )


def query_by_cpc(*columns):
    """Applications with at least one CPC symbol in the CPC list."""
    return (
        db.query(*columns)
        .select_from(TLS201_APPLN)
        .join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
        .filter(symbol_prefix_condition(TLS224_APPLN_CPC.cpc_class_symbol, SECTOR_CPC_PREFIXES))
    )

In [103]:
appln_id = TLS201_APPLN.appln_id.label("appln_id")

sd_ids = (
    query_by_keywords(appln_id)
    .union(query_by_cpc(appln_id))
    .subquery()
)

q_sd = (
    db.query(
        *APPLN_COLUMNS,
        TLS202_APPLN_TITLE.appln_title,
        TLS206_PERSON.psn_name,
        TLS206_PERSON.psn_sector,
        TLS206_PERSON.person_ctry_code,
    )
    .select_from(TLS201_APPLN)
    .join(sd_ids, TLS201_APPLN.appln_id == sd_ids.c.appln_id)
    .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
    .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
    .outerjoin(
        TLS202_APPLN_TITLE,
        TLS201_APPLN.appln_id == TLS202_APPLN_TITLE.appln_id
    )
    .filter(TLS207_PERS_APPLN.applt_seq_nr > 0)  # Retain applicants only (if desired)
    .distinct()
    .order_by(TLS201_APPLN.appln_filing_date)
)

df_sd = patstat.df(q_sd)

In [104]:
df_sd

,appln_id,appln_auth,appln_nr,appln_kind,appln_filing_date,appln_title,psn_name,psn_sector,person_ctry_code
0,21651327,GB,1258422,A,1922-05-04,Improvements in windmills and wind-driven elec...,JOHN HALIFAX GARTY,INDIVIDUAL,
1,21622371,GB,1178225,A,1925-05-06,Method of and arrangement for preventing the f...,HERMANN FOETTINGER,INDIVIDUAL,
2,21813502,GB,1700425,A,1926-02-22,Improvements in or relating to windmills or wi...,HENRY TURNER HOWITT,INDIVIDUAL,
3,21813502,GB,1700425,A,1926-02-22,Improvements in or relating to windmills or wi...,ANDREW MCBAIN,INDIVIDUAL,
4,21813502,GB,1700425,A,1926-02-22,Improvements in or relating to windmills or wi...,WALTER JAMES MCBAIN,INDIVIDUAL,
...,...,...,...,...,...,...,...,...,...
4015,598825669,JP,2023120014,A,2023-07-24,"ADJUSTABLE PILE HOLDING SYSTEM, MARINE VESSEL ...",ITREC BV,UNKNOWN,
4016,596998826,EP,23189493,A,2023-08-03,"A METHOD FOR BUILDING AN OFFSHORE OIL, GAS OR ...",Iv-Groep B.V.,UNKNOWN,NL
4017,600038594,CN,202311010448,A,2023-08-11,"Floating platform integrating wave resistance,...",JIANGSU UNIVERSITY OF SCIENCE AND TECHNOLOGY,UNIVERSITY,
4018,598686347,AU,2023219924,A,2023-08-24,"Self-powered, self-propelled compute grid with...",LONE GULL HOLDINGS,COMPANY,US


In [105]:
def applicant_type(psn_sector):
    """Translate PATSTAT psn_sector into standardized sector types."""
    if pd.isna(psn_sector) or not psn_sector:
        return "unknown"
        
    s = str(psn_sector).upper()
    if "UNIVERSITY" in s:
        return "university"
    if "COMPANY" in s:
        return "firm"
    if "GOV" in s:
        return "government / non-profit"
    if "HOSPITAL" in s:
        return "hospital"
    if "INDIVIDUAL" in s:
        return "individual"
    return "unknown"

    
def consolidated_names(psn_names):
    names = sorted(set(psn_names))
    "Dictionary {PATSTAT name: consolidated name}, with the method chosen in NAME_METHOD."
    return {n: clean_with_cleanco(n) for n in names}



def prepare_applicants(raw):
    
    "From (appln_id, year, person) rows to one row per (appln_id, applicant) with type and country."
    df = raw.copy()

    # 1. the consolidated name of each applicant
    df["applicant"] = df["psn_name"].map(consolidated_names(df["psn_name"]))

    # 2. the type of applicant
    df["type"] = df["psn_sector"].map(applicant_type)

    # 3. one country per applicant: the most frequent known one
    df["ctry"] = df["person_ctry_code"].replace("", np.nan)
    known = df.dropna(subset=["ctry"]).drop_duplicates(["appln_id", "applicant", "ctry"])
    main_country = known.groupby("applicant")["ctry"].agg(lambda c: c.value_counts().index[0])
    df["ctry"] = df["applicant"].map(main_country)

    # 4. add year
    df["year"] = pd.to_datetime(df["appln_filing_date"]).dt.year
    
    # 5. one row per (patent, applicant)
    return df[["appln_id", "year", "applicant", "type", "ctry"]].drop_duplicates(["appln_id", "applicant"])

In [106]:
applicants_df = prepare_applicants(df_sd)

In [107]:
applicants_df

,appln_id,year,applicant,type,ctry
0,21651327,1922,JOHN HALIFAX GARTY,individual,
1,21622371,1925,HERMANN FOETTINGER,individual,
2,21813502,1926,HENRY TURNER HOWITT,individual,
3,21813502,1926,ANDREW MCBAIN,individual,
4,21813502,1926,WALTER JAMES MCBAIN,individual,
...,...,...,...,...,...
4015,598825669,2023,ITREC,unknown,NL
4016,596998826,2023,IV GROEP,unknown,NL
4017,600038594,2023,JIANGSU UNIVERSITY OF SCIENCE AND TECHNOLOGY,university,
4018,598686347,2023,LONE GULL HOLDINGS,firm,US


## Harmonization Function

In [108]:
!pip install sentence-transformers


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip


In [146]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

gold = pd.read_csv(GOLD_STANDARD_FILE).rename(
    columns={"comp": "name", "Group": "gold_group"}
)
gold["name"] = gold["name"].astype(str)

# Drop duplicate names while keeping their corresponding gold_group label
unique_gold = gold.drop_duplicates(subset=["name"]).reset_index(drop=True)

SBERT_MODEL = "graziasveva93/sbert-adapt-ub"
model = SentenceTransformer(SBERT_MODEL)

# Encode in the exact order of unique_gold
gold_embeddings = model.encode(
    unique_gold["name"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# 1. Compute full N x N cosine similarity matrix
sim_matrix = gold_embeddings @ gold_embeddings.T

# 2. Extract upper triangle indices (k=1 excludes self-similarity of 1.0 and duplicate pairs)
i, j = np.triu_indices(len(unique_gold), k=1)

pairs_df = pd.DataFrame({
    "name_1": unique_gold["name"].values[i],
    "name_2": unique_gold["name"].values[j],
    "group_1": unique_gold["gold_group"].values[i],
    "group_2": unique_gold["gold_group"].values[j],
    "cosine_sim": sim_matrix[i, j],
})

# Flag pairs that belong to the same gold_group
pairs_df["same_group"] = pairs_df["group_1"] == pairs_df["group_2"]

# Group by the label to get summary statistics per gold_group (intra-group similarity)
intra_group_stats = (
    pairs_df[pairs_df["same_group"]]
    .groupby("group_1")["cosine_sim"]
    .agg(pair_count="count", mean_sim="mean", min_sim="min", max_sim="max", std_sim="std")
    .reset_index()
    .rename(columns={"group_1": "gold_group"})
)

print(intra_group_stats)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/100 [00:00<?, ?it/s]

     gold_group  pair_count  mean_sim   min_sim   max_sim   std_sim
0             1        1081  0.792484  0.273350  0.994304  0.138630
1             2          15  0.947395  0.855963  0.994366  0.037295
2             3          15  0.661656  0.379690  0.969626  0.229275
3             4        5356  0.705538 -0.114928  0.995147  0.177453
4             7        1326  0.588674  0.004002  0.992722  0.171207
..          ...         ...       ...       ...       ...       ...
181         494        1485  0.609116 -0.182192  1.000000  0.219620
182         496         105  0.869579  0.547519  0.990560  0.088289
183         497           1  0.474511  0.474511  0.474511       NaN
184         498          10  0.665866  0.358378  0.991451  0.229213
185         499           3  0.767611  0.641541  0.921145  0.141811

[186 rows x 6 columns]


In [152]:
from sklearn.metrics import average_precision_score, roc_auc_score

y_true = pairs_df["same_group"].astype(int)
y_scores = pairs_df["cosine_sim"]

# 1. PR-AUC (Best metric when same_group=True is rare)
pr_auc = average_precision_score(y_true, y_scores)

# 2. ROC-AUC (Probability that a random positive pair scores higher than a random negative pair)
roc_auc = roc_auc_score(y_true, y_scores)

print(f"PR-AUC:  {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

PR-AUC:  0.1973
ROC-AUC: 0.8056


In [129]:
gold['gold_group'].value_counts(ascending=True)

gold_group
41       1
29       1
30       1
32       1
33       1
      ... 
4      104
478    111
155    119
269    179
212    198
Name: count, Length: 481, dtype: int64

<u>Note:</u>

Using train_test_split and training with a classifier is not viable since for some labels we only have one example.
stratify=y requires every class in gold["gold_group"]=y to appear at least twice (so it can put at least one example in the training set and one in the test set).
At least one of our gold_group labels only has 1 row in gold.

So we rely on direct cosine similarity comparison:

In [143]:
# Find the single closest match in `gold` for each applicant
hits = semantic_search(
    torch.tensor(app_embeddings),
    torch.tensor(gold_embeddings),
    top_k=1
)

# Extract the matched row index and cosine similarity score
best_idx = [hit[0]["corpus_id"] for hit in hits]
best_scores = [hit[0]["score"] for hit in hits]

# Attach the matched gold name, gold_group, and similarity score
applicants_df["matched_gold_name"] = gold["name"].iloc[best_idx].values
applicants_df["predicted_group"] = gold["gold_group"].iloc[best_idx].values
applicants_df["similarity_score"] = best_scores

In [145]:
applicants_df[applicants_df['similarity_score']>0.95]

,appln_id,year,applicant,type,ctry,matched_gold_name,predicted_group,similarity_score
0,21651327,1922,JOHN HALIFAX GARTY,individual,,NOKIA SIEMENS NETWORKS,2,1.000000
1,21622371,1925,HERMANN FOETTINGER,individual,,NOKIA COMMUNICATION,1,1.000000
2,21813502,1926,HENRY TURNER HOWITT,individual,,NOKIA SIEMENS NETWORKS,2,1.000000
3,21813502,1926,ANDREW MCBAIN,individual,,NOKIA COMMUNICATION,1,1.000000
4,21813502,1926,WALTER JAMES MCBAIN,individual,,NOKIA SIEMENS NETWORKS,2,1.000000
...,...,...,...,...,...,...,...,...
4010,594474963,2023,MACGREGOR NORWAY,firm,NO,NOKIA COMMUNICATION,1,1.000000
4011,595317898,2023,ITREC,firm,NL,NOKIA MECHATRONICS,1,1.000001
4012,596565017,2023,ITREC,firm,NL,NOKIA MECHATRONICS,1,1.000001
4015,598825669,2023,ITREC,unknown,NL,NOKIA CONSUMER ELECTRONICS,1,1.000000


## Evaluating harmonization

How good is a harmonisation method? 

To know, we need names for which the right 
answer is already known: a gold standard.

The data. The file PATSTAT_500_label_datav4.csv (in the same folder as this notebook) contains 
applicant names as they appear in PATSTAT, grouped by hand. It has two columns:

In [89]:
from math import comb
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering

SBERT_MODEL = "graziasveva93/sbert-adapt-ub"
model = SentenceTransformer(SBERT_MODEL)


def pairs_inside(group_sizes):
    "Number of pairs of names inside groups of the given sizes: n(n-1)/2 for each group."
    return sum(comb(int(n), 2) for n in group_sizes)


def precision_recall_f1(gold_groups, predicted_groups):
    "Pairwise precision, recall and F1 of a predicted grouping against the gold groups."
    df = pd.DataFrame({"gold": list(gold_groups), "pred": list(predicted_groups)})
    tp = pairs_inside(df.groupby(["gold", "pred"]).size())         # merged by you and in the gold standard
    tp_fp = pairs_inside(df.groupby("pred").size())                # all the pairs you merged
    tp_fn = pairs_inside(df.groupby("gold").size())                # all the pairs that belong together
    precision = tp / tp_fp if tp_fp else 1.0
    recall = tp / tp_fn
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"groups found": df["pred"].nunique(), "precision": precision, "recall": recall, "F1": f1}


def cluster_names_sbert(names, threshold=0.80):
    """Clean names with cleanco, encode with SBERT, and cluster via average-linkage."""
    series = pd.Series(names).fillna("").astype(str)
    unique_raw = series.unique()
    cleaned_unique = [clean_with_cleanco(n) for n in unique_raw]

    embs = model.encode(
        cleaned_unique,
        batch_size=32,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    sim_matrix = embs @ embs.T
    dist_matrix = np.clip(1.0 - sim_matrix, 0.0, 2.0)

    clustering = AgglomerativeClustering(
        n_clusters=None,
        metric="precomputed",
        linkage="average",
        distance_threshold=1.0 - threshold,
    )
    unique_labels = clustering.fit_predict(dist_matrix)
    label_map = dict(zip(unique_raw, unique_labels))
    return series.map(label_map).values


def my_harmonisation(app_df, gold_df):
    """
    1. Finds the F1-optimal clustering threshold on `gold_df`.
    2. Clusters `app_df['applicant']` with that threshold and returns
       an array of canonical applicant names aligned with `app_df`.
    """
    # Step 1: Tune best threshold on the gold standard
    best_thresh = 0.80
    best_f1 = -1.0
    for thresh in [0.72, 0.75, 0.78, 0.80, 0.82, 0.85, 0.88, 0.90]:
        pred_gold = cluster_names_sbert(gold_df["name"], threshold=thresh)
        f1 = precision_recall_f1(gold_df["gold_group"], pred_gold)["F1"]
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh

    # Step 2: Cluster `app_df` using the winning threshold
    clusters = cluster_names_sbert(app_df["applicant"], threshold=best_thresh)

    # Map each cluster to its most frequent spelling in `app_df`
    temp = pd.DataFrame({"applicant": app_df["applicant"].values, "cluster": clusters})
    canonical_map = temp.groupby("cluster")["applicant"].agg(lambda s: s.value_counts().index[0])

    return temp["cluster"].map(canonical_map).values

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]